# SHIPP Stage 7B — Silver routes

`ors_route_responses` → `silver_routes`. **Grain:** one row per `(origin_key, dest_key, profile)`,
from the latest successful response. `route_status` = `OK` or `NO_ROUTE`; distance is never invented.
Logic: `data_pipeline/silver/routes.py` (unit-tested). Rebuilt from Bronze each run → idempotent.

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `silver_routes`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
import config.tables as t

print("Loaded config file:", t.__file__)
print("SILVER_CANDIDATE_PAIRS:", t.SILVER_CANDIDATE_PAIRS)
print("BRONZE_ROUTE_RESPONSES:", t.BRONZE_ROUTE_RESPONSES)
print("SILVER_ROUTES:", t.SILVER_ROUTES)
print("PIPELINE_RUN_LOG:", t.PIPELINE_RUN_LOG)

print("PASS — centralized table registry loaded")

In [0]:
from data_pipeline.silver.routes import (
    ROUTE_COLUMNS,
    parse_route_responses,
)

# ------------------------------------------------------------
# Dependency gates
# ------------------------------------------------------------

assert table_exists(BRONZE_ROUTE_RESPONSES), (
    f"BLOCKED: missing {BRONZE_ROUTE_RESPONSES}. "
    "Run 31_ors_route_enrichment first."
)

assert table_exists(SILVER_CANDIDATE_PAIRS), (
    f"BLOCKED: missing {SILVER_CANDIDATE_PAIRS}. "
    "Run 30_silver_candidate_pairs first."
)

EXPECTED_COLUMNS = ROUTE_COLUMNS + [
    "silver_processed_at"
]


# ------------------------------------------------------------
# Bronze ORS → trusted Silver routes
# ------------------------------------------------------------

routes_df = (
    parse_route_responses(
        spark.table(BRONZE_ROUTE_RESPONSES)
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp(),
    )
)

row_count = routes_df.count()

print("Silver route candidate rows:", row_count)

display(
    routes_df.orderBy(
        "dest_key",
        "distance_km",
    )
)

In [0]:
# ------------------------------------------------------------
# Quality gate
# ------------------------------------------------------------

run_quality_gate(
    {
        "duplicate route key": duplicate_key_count(
            routes_df,
            "origin_key",
            "dest_key",
            "profile",
        ),

        "invalid route status": routes_df.filter(
            ~F.col("route_status").isin(
                "OK",
                "NO_ROUTE",
            )
        ).count(),

        "negative distance": routes_df.filter(
            F.col("distance_km") < 0
        ).count(),

        "negative duration": routes_df.filter(
            F.col("duration_min") < 0
        ).count(),

        "OK without distance": routes_df.filter(
            (F.col("route_status") == "OK")
            & F.col("distance_km").isNull()
        ).count(),

        "NO_ROUTE with distance": routes_df.filter(
            (F.col("route_status") == "NO_ROUTE")
            & F.col("distance_km").isNotNull()
        ).count(),
    },
    row_count=row_count,
    allow_empty=True,
)


# ------------------------------------------------------------
# Persist Silver
# ------------------------------------------------------------

written = write_overwrite(
    routes_df,
    SILVER_ROUTES,
)

assert written == row_count, (
    f"WRITE VALIDATION FAILED: "
    f"wrote {written}, expected {row_count}"
)

assert_exact_schema(
    SILVER_ROUTES,
    EXPECTED_COLUMNS,
)

print(
    "PASS — Silver routes written:",
    written,
)

In [0]:
# ------------------------------------------------------------
# Final coverage validation
# ------------------------------------------------------------

candidate_route_keys = (
    spark.table(SILVER_CANDIDATE_PAIRS)
    .select(
        "origin_key",
        "dest_key",
        "profile",
    )
    .distinct()
)

persisted_route_keys = (
    spark.table(SILVER_ROUTES)
    .select(
        "origin_key",
        "dest_key",
        "profile",
    )
    .distinct()
)

candidate_key_count = candidate_route_keys.count()
persisted_key_count = persisted_route_keys.count()

uncovered = (
    candidate_route_keys
    .join(
        persisted_route_keys,
        [
            "origin_key",
            "dest_key",
            "profile",
        ],
        "left_anti",
    )
)

uncovered_count = uncovered.count()

print("Candidate route keys :", candidate_key_count)
print("Persisted route keys :", persisted_key_count)
print("Uncovered route keys :", uncovered_count)

if uncovered_count > 0:
    display(uncovered)

assert uncovered_count == 0, (
    f"FAILED: {uncovered_count} candidate route keys "
    "do not have a Silver route. "
    "Rerun Stage 6 ORS enrichment."
)

print(
    "PASS — every candidate route key "
    "is covered by silver_routes."
)


summary = {
    "table": SILVER_ROUTES,
    "rows": written,
    "candidate_route_keys": candidate_key_count,
    "uncovered_pairs": uncovered_count,
    "status": "PASS",
}

print(summary)

dbutils.notebook.exit(
    json.dumps(summary)
)